# 5.4 Multiple Linear Regression

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 5: Linear Regression

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-05-linear-regression/05.04_multiple-linear-regression.ipynb)

Companion code for Section 5.4 of the book. Run the cells in order from top to bottom.

**Example 5.4.1**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.datasets import fetch_california_housing

# Load California Housing dataset
housing = fetch_california_housing(as_frame=True)
df = housing.frame
print('Features:', housing.feature_names)
print('Shape:', df.shape)
print(df.describe().round(3))

X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']

# Train-test split + scaling
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# Fit multiple linear regression
model = LinearRegression()
model.fit(X_train_s, y_train)

# Predictions
y_pred_train = model.predict(X_train_s)
y_pred_test  = model.predict(X_test_s)

print('\n=== Multiple Linear Regression Results ===')
print(f'Intercept β₀:  {model.intercept_:.4f}')
print('\nCoefficients (standardised features → comparable magnitudes):')
coef_df = pd.DataFrame({
    'feature':     housing.feature_names,
    'coefficient': model.coef_,
    'abs_impact':  np.abs(model.coef_)
}).sort_values('abs_impact', ascending=False)
print(coef_df.round(4).to_string(index=False))

print(f'\nTrain R²:   {r2_score(y_train, y_pred_train):.4f}')
print(f'Test  R²:   {r2_score(y_test, y_pred_test):.4f}')
print(f'Train RMSE: {np.sqrt(mean_squared_error(y_train, y_pred_train)):.4f}')
print(f'Test  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.4f}')

# Cross-validation
cv_scores = cross_val_score(LinearRegression(), scaler.fit_transform(X), y,
                             cv=5, scoring='r2')
print(f'\n5-Fold CV R²: {cv_scores.round(4)}')
print(f'Mean ± Std:   {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

# ── Comprehensive diagnostic plots ────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Multiple Linear Regression Diagnostics - California Housing',
            fontsize=14, fontweight='bold')

# 1. Actual vs Predicted
ax = axes[0,0]
ax.scatter(y_test, y_pred_test, alpha=0.3, s=8, color='#1976D2')
lims = [min(y_test.min(), y_pred_test.min()), max(y_test.max(), y_pred_test.max())]
ax.plot(lims, lims, 'r--', lw=2)
ax.set_xlabel('Actual House Value ($100k)'); ax.set_ylabel('Predicted')
ax.set_title(f'Actual vs Predicted (R²={r2_score(y_test,y_pred_test):.3f})')

# 2. Residuals vs Fitted
residuals = y_test - y_pred_test
ax = axes[0,1]
ax.scatter(y_pred_test, residuals, alpha=0.3, s=8, color='#E53935')
ax.axhline(0, color='black', lw=1.5, linestyle='--')
ax.set_xlabel('Fitted Values'); ax.set_ylabel('Residuals')
ax.set_title('Residuals vs Fitted')

# 3. Residual histogram
ax = axes[0,2]
ax.hist(residuals, bins=50, color='#1976D2', alpha=0.7, edgecolor='white')
ax.set_xlabel('Residuals'); ax.set_ylabel('Frequency')
ax.set_title('Residual Distribution')

# 4. Q-Q plot
from scipy import stats
ax = axes[1,0]
stats.probplot(residuals, plot=ax)
ax.set_title('Q-Q Plot (Normality Check)')

# 5. Coefficient bar chart
ax = axes[1,1]
colors = ['#1976D2' if c > 0 else '#E53935' for c in coef_df.coefficient]
ax.barh(coef_df.feature, coef_df.coefficient, color=colors, edgecolor='white')
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('Standardised Coefficient'); ax.set_title('Feature Coefficients')

# 6. Scale-Location plot
ax = axes[1,2]
ax.scatter(y_pred_test, np.sqrt(np.abs(residuals)), alpha=0.3, s=8, color='#43A047')
ax.set_xlabel('Fitted Values'); ax.set_ylabel('√|Residuals|')
ax.set_title('Scale-Location Plot')

plt.tight_layout()
plt.savefig('multiple_regression_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

**Example 5.4.2**

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd
import numpy as np

# Compute VIF for each feature
def compute_vif(X_df):
    vif_data = pd.DataFrame()
    vif_data['Feature'] = X_df.columns
    vif_data['VIF']     = [variance_inflation_factor(X_df.values, i)
                            for i in range(X_df.shape[1])]
    vif_data['Status']  = vif_data['VIF'].apply(
        lambda v: 'OK' if v<5 else ('Warning' if v<10 else 'SEVERE'))
    return vif_data.sort_values('VIF', ascending=False)

from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing(as_frame=True)
X_vif = housing.frame.drop('MedHouseVal', axis=1)

print('Variance Inflation Factors:')
print(compute_vif(X_vif).round(3).to_string(index=False))